In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon
import random
from google.colab import files

# 1. INITIALIZE ENVIRONMENT & COORDINATES
volcano_lon, volcano_lat = 108.0583, -7.2500
crs_gps = "EPSG:4326"
crs_metric = "EPSG:23848"

center_point = gpd.GeoDataFrame(geometry=[Point(volcano_lon, volcano_lat)], crs=crs_gps).to_crs(crs_metric)
center_geom = center_point.geometry.iloc[0]

# 2. GENERATE COMPLIANT HAZARD ZONES (KRB 1, 2, 3)
radius_krb3 = np.sqrt((439.7 * 10000) / np.pi)
radius_krb2 = np.sqrt((1050.5 * 10000) / np.pi)
radius_krb1 = np.sqrt((5431.5 * 10000) / np.pi)

poly_krb3 = center_geom.buffer(radius_krb3)
poly_krb2 = center_geom.buffer(radius_krb2).difference(poly_krb3)
poly_krb1 = center_geom.buffer(radius_krb1).difference(center_geom.buffer(radius_krb2))

hazard_gdf = gpd.GeoDataFrame({
    'KRB_Level': ['KRB 3', 'KRB 2', 'KRB 1'],
    'Risk_Score': [3, 2, 1],
    'Description': ['High Risk (Lava/Lahar Flow Paths)', 'Medium Risk (Potential Trajectory)', 'Low Risk (Ash/Lapilli Alert)']
}, geometry=[poly_krb3, poly_krb2, poly_krb1], crs=crs_metric)

# 3. GENERATE THE 13 ADMINISTRATIVE SUB-DISTRICT BOUNDARIES
kecamatan_names = [
    'Padakembang', 'Sukaratu', 'Sukurame', 'Tanjungjaya', 'Singaparna',
    'Sariwangi', 'Mangunreja', 'Leuwisari', 'Indihiang', 'Cisayong',
    'Cipedes', 'Cigalontang', 'Bungursari'
]

angles = np.linspace(0, 2 * np.pi, len(kecamatan_names) + 1)
kec_polygons = []

for i, name in enumerate(kecamatan_names):
    p1 = center_geom
    p2 = Point(center_geom.x + radius_krb1 * 1.5 * np.cos(angles[i]), center_geom.y + radius_krb1 * 1.5 * np.sin(angles[i]))
    p3 = Point(center_geom.x + radius_krb1 * 1.5 * np.cos(angles[i+1]), center_geom.y + radius_krb1 * 1.5 * np.sin(angles[i+1]))
    wedge = Polygon([p1, p2, p3])
    kec_polygons.append(wedge)

np.random.seed(19)
pop_densities = [1200 if name == 'Padakembang' else np.random.randint(400, 950) for name in kecamatan_names]

subdistricts_gdf = gpd.GeoDataFrame({
    'Kecamatan': kecamatan_names,
    'Base_Pop_Density_SqKm': pop_densities
}, geometry=kec_polygons, crs=crs_metric)

# 4. EXECUTE SPATIAL OVERLAY INTERSECTION
intersection_gdf = gpd.overlay(subdistricts_gdf, hazard_gdf, how='intersection')
intersection_gdf['Exposed_Area_Ha'] = intersection_gdf.geometry.area / 10000
intersection_gdf['Exposed_Population_Est'] = ((intersection_gdf['Exposed_Area_Ha'] / 100) * intersection_gdf['Base_Pop_Density_SqKm']).astype(int)

# ==========================================
# THE SENIOR FLEX: DENSE POINT SAMPLING FOR VISUAL COHESION
# ==========================================
print("Generating dense spatial point cloud for Looker Studio fill optimization...")

point_records = []
random.seed(42)

# Loop through each unique intersection shape and pack it with coordinates
for idx, row in intersection_gdf.iterrows():
    poly = row.geometry
    if poly.is_empty:
        continue

    min_x, min_y, max_x, max_y = poly.bounds
    points_inserted = 0
    # Higher number creates a denser, smoother visual fill
    target_points = 80

    attempts = 0
    while points_inserted < target_points and attempts < 2000:
        attempts += 1
        rand_point = Point(random.uniform(min_x, max_x), random.uniform(min_y, max_y))

        if poly.contains(rand_point):
            # Convert point back to GPS coordinate system
            pt_gdf = gpd.GeoDataFrame(geometry=[rand_point], crs=crs_metric).to_crs(crs_gps)
            gps_geom = pt_gdf.geometry.iloc[0]

            point_records.append({
                'Kecamatan': row['Kecamatan'],
                'KRB_Level': row['KRB_Level'],
                'Description': row['Description'],
                'Exposed_Area_Ha': round(row['Exposed_Area_Ha'], 2),
                'Base_Pop_Density_SqKm': row['Base_Pop_Density_SqKm'],
                'Exposed_Population_Est': row['Exposed_Population_Est'],
                'Latitude': gps_geom.y,
                'Longitude': gps_geom.x
            })
            points_inserted += 1

dense_portfolio_df = pd.DataFrame(point_records)

# Export and Download
csv_filename = "Galunggung_Dense_Cloud_Fixed.csv"
dense_portfolio_df.to_csv(csv_filename, index=False)
files.download(csv_filename)

print("✅ Dense spatial dataset downloaded and optimized for Looker Studio UI!")

In [ ]:
import fiona
from google.colab import files

# Enable KML support in GeoPandas
fiona.drvsupport.supported_drivers['KML'] = 'rw'

# Convert to standard GPS projection required by Google Maps
kml_gdf = intersection_gdf.to_crs("EPSG:4326").copy()

# Keep only the essential columns for the map popup (My Maps struggles with too much data)
kml_gdf = kml_gdf[['Kecamatan', 'KRB_Level', 'Description', 'Exposed_Population_Est', 'geometry']]

# Convert population to string to ensure safe KML export
kml_gdf['Exposed_Population_Est'] = kml_gdf['Exposed_Population_Est'].astype(str)

# Export to KML
kml_filename = "Galunggung_Hazard_Zones.kml"
kml_gdf.to_file(kml_filename, driver='KML')

# Download the file
files.download(kml_filename)
print("✅ KML File downloaded and ready for Google My Maps!")

In [ ]:
import fiona
from google.colab import files

# Enable KML support in GeoPandas
fiona.drvsupport.supported_drivers['KML'] = 'rw'

# ---- LAYER 1: FULL ADMINISTRATIVE BOUNDARIES (KECAMATAN) ----
print("Preparing administrative boundary layer...")
kec_kml = subdistricts_gdf.to_crs("EPSG:4326").copy()
# Keep only administrative metadata
kec_kml = kec_kml[['Kecamatan', 'Base_Pop_Density_SqKm', 'geometry']]
kec_kml['Base_Pop_Density_SqKm'] = kec_kml['Base_Pop_Density_SqKm'].astype(str)

kec_filename = "Galunggung_Kecamatan_Boundaries.kml"
kec_kml.to_file(kec_filename, driver='KML')
files.download(kec_filename)


# ---- LAYER 2: VOLCANIC HAZARD ZONES (KRB 1, 2, 3) ----
print("Preparing environmental hazard layer...")
haz_kml = hazard_gdf.to_crs("EPSG:4326").copy()
# Keep only hazard metadata
haz_kml = haz_kml[['KRB_Level', 'Description', 'geometry']]

haz_filename = "Galunggung_Hazard_Zones.kml"
haz_kml.to_file(haz_filename, driver='KML')
files.download(haz_filename)

print("✅ Both GIS layers successfully generated and downloaded!")